# Applied Data Science Capstone — SpaceX Falcon 9 Landing Prediction

**Student:** Harshit Singh Jamwal  
**Course:** Applied Data Science Capstone

Run every cell from top to bottom. This notebook retrieves current public data;
results may change as the source data changes. Do not report metrics until you run the notebook.


## 1. Setup

In [ ]:
from pathlib import Path
import sys, requests, pandas as pd, numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
for p in ["data/raw","data/processed","outputs/figures","outputs/maps"]:
    (ROOT/p).mkdir(parents=True, exist_ok=True)


## 2. SpaceX API data collection

In [ ]:
from collect_spacex_data import main
main()
launches = pd.read_csv(ROOT / "data/raw/spacex_launches.csv")
display(launches.head())
print("Rows:", len(launches))


## 3. Web scraping

In [ ]:
# Demonstration: retrieve publicly listed Falcon 9 launch tables.
# This external page can change; if it fails, record the error and continue with API data.
try:
    tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches")
    print("Tables retrieved:", len(tables))
    if tables:
        display(tables[0].head())
        tables[0].to_csv(ROOT / "data/raw/web_launch_table.csv", index=False)
except Exception as e:
    print("Web scraping request failed:", e)


## 4. Data wrangling

In [ ]:
df = launches.copy()
df["date_utc"] = pd.to_datetime(df["date_utc"], errors="coerce", utc=True)
df["year"] = pd.to_numeric(df["year"], errors="coerce")
for col in ["landing_attempt","landing_success","reused"]:
    df[col] = df[col].map({True:1, False:0, "True":1, "False":0})
display(df.isna().sum().to_frame("missing values"))


## 5. EDA with charts

In [ ]:
yearly = df.groupby("year").size()
ax = yearly.plot(kind="bar", figsize=(12,4), title="Launches by year")
ax.set_xlabel("Year"); ax.set_ylabel("Launch count"); plt.tight_layout()
plt.savefig(ROOT/"outputs/figures/launches_by_year.png"); plt.show()

outcomes = df["landing_success"].map({1:"Success",0:"Failure"}).fillna("Unknown").value_counts()
ax = outcomes.plot(kind="bar", figsize=(7,4), title="Recorded landing outcomes")
ax.set_xlabel("Outcome"); ax.set_ylabel("Count"); plt.tight_layout()
plt.savefig(ROOT/"outputs/figures/landing_outcomes.png"); plt.show()


## 6. EDA with SQL

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
df.to_sql("launches", con, index=False)
queries = {
 "Launches by year": "SELECT year, COUNT(*) AS launches FROM launches GROUP BY year ORDER BY year",
 "Landing outcomes": "SELECT landing_success, COUNT(*) AS total FROM launches GROUP BY landing_success",
 "Launches by site": "SELECT launchpad, COUNT(*) AS launches FROM launches GROUP BY launchpad ORDER BY launches DESC"
}
for title, q in queries.items():
    print(title)
    display(pd.read_sql_query(q, con))


## 7. Folium launch-site map

In [ ]:
pads = requests.get("https://api.spacexdata.com/v4/launchpads", timeout=45).json()
import folium
m = folium.Map(location=[28.5,-80.6], zoom_start=3)
for p in pads:
    if p.get("latitude") is not None and p.get("longitude") is not None:
        folium.Marker([p["latitude"],p["longitude"]],
                      popup=p.get("full_name","Launch site")).add_to(m)
m.save(str(ROOT/"outputs/maps/launch_sites.html"))
m


## 8. Prepare features for prediction

In [ ]:
rockets = requests.get("https://api.spacexdata.com/v4/rockets", timeout=45).json()
payloads = requests.get("https://api.spacexdata.com/v4/payloads", timeout=45).json()
rocket_names = {r["_id"]:r.get("name") for r in rockets}
payload_mass = {p["_id"]:p.get("mass_kg") for p in payloads}
model = df.copy()
model["rocket_name"] = model["rocket"].map(rocket_names)
model["payload_mass_kg"] = model["payload_ids"].apply(
    lambda ids: sum(payload_mass.get(i) or 0 for i in ids) if isinstance(ids,list) else np.nan)
model = model[(model.landing_attempt == 1)].dropna(subset=["landing_success","year","launchpad"])
model["landing_success"] = model["landing_success"].astype(int)
display(model[["flight_number","year","launchpad","rocket_name","payload_mass_kg","landing_success"]].head())
print("Labelled landing attempts:",len(model))


## 9. Classification models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

features = ["flight_number","year","launchpad","rocket_name","payload_mass_kg"]
X, y = model[features], model["landing_success"]
if y.nunique() < 2 or len(model) < 10:
    raise ValueError("Not enough labelled examples/classes for a meaningful model.")
numeric = ["flight_number","year","payload_mass_kg"]
categorical = ["launchpad","rocket_name"]
prep = ColumnTransformer([
 ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric),
 ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                  ("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical)
])
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
models = {
 "Logistic Regression":LogisticRegression(max_iter=2000),
 "SVM":SVC(),
 "Decision Tree":DecisionTreeClassifier(max_depth=5,random_state=42),
 "KNN":KNeighborsClassifier(n_neighbors=5)
}
fitted, scores = {}, []
for name, estimator in models.items():
    pipe = Pipeline([("prep",prep),("model",estimator)])
    pipe.fit(Xtr,ytr)
    pred = pipe.predict(Xte)
    fitted[name] = pipe
    scores.append({"model":name,"accuracy":accuracy_score(yte,pred)})
scores = pd.DataFrame(scores).sort_values("accuracy",ascending=False)
display(scores)
scores.to_csv(ROOT/"outputs/model_comparison.csv",index=False)


## 10. Confusion matrix and evaluation

In [ ]:
best_name = scores.iloc[0]["model"]
best = fitted[best_name]
pred = best.predict(Xte)
print("Best model by test accuracy:",best_name)
print(classification_report(yte,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(yte,pred)
plt.title("Confusion matrix — " + best_name)
plt.tight_layout()
plt.savefig(ROOT/"outputs/figures/confusion_matrix.png")
plt.show()


## 11. Save dashboard data

In [ ]:
model[["launchpad","payload_mass_kg","landing_success","flight_number","year"]].to_csv(
    ROOT/"data/processed/launches_model.csv",index=False)
print("Saved dashboard data. Run: python src/dashboard.py")


## 12. Conclusion — complete using your real outputs

Record the actual model accuracy and confusion matrix, describe patterns visible in your charts,
explain the best model, and discuss limitations. Do not invent scores or conclusions.
